In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [7]:
df = pd.read_csv("spotify_most_streamed_songs.csv")
print(df.info())
print(df.isna().sum())
print(df.duplicated().sum())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  in_shazam_charts      903 non-null    str  
 14  bpm                   953 non-null    int64
 15  key                   858 non-null    str  
 16  mode               

In [8]:
df["streams"] = pd.to_numeric(df["streams"], errors="coerce")


df = df.drop_duplicates(
    subset=["track_name", "artist(s)_name"]
).copy()

features = ["bpm", "danceability_%", "valence_%", "energy_%", "acousticness_%", "instrumentalness_%", "liveness_%", "speechiness_%"]



df = df.dropna(subset=features + ["streams"]).copy()
df = df.reset_index(drop=True)

#I am separating features form the target
X = df[features].copy()
y = np.log1p(df["streams"])

# 9. Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=features,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=features,
    index=X_test.index
)


df.to_csv("spotify_regression_cleaned.csv", index=False)


print("Training size:", X_train_scaled.shape)
print("Testing size:", X_test_scaled.shape)
print(X_train_scaled.head())

Training size: (758, 8)
Testing size: (190, 8)
          bpm  danceability_%  valence_%  energy_%  acousticness_%  \
109 -0.529374       -0.401508  -0.189958  0.219488       -1.018789   
77   0.289463        0.274359  -1.179421 -1.053304       -0.980540   
549 -1.455016       -0.266334   1.444806 -0.144167       -0.789297   
798 -1.597422       -1.618068   0.369303 -1.598787        1.888115   
82   0.253861       -0.266334  -0.663179 -1.841223        1.888115   

     instrumentalness_%  liveness_%  speechiness_%  
109           -0.179790   -0.376087      -0.730486  
77            -0.179790    0.692749      -0.122683  
549           -0.179790    1.303512      -0.629186  
798           -0.056011    2.296003       2.815032  
82            -0.179790    1.303512      -0.122683  
